# Comparison of Ranges Data to SPARQL Results

In [1]:
import pandas as pd
import rdflib
from rdflib.plugins.sparql.processor import SPARQLResult

---

## helper functions

In [2]:
def sparql_results_to_df(results:SPARQLResult) -> pd.DataFrame:
    """
    Export results from an rdflib SPARQL query into a `pandas.DataFrame`,
    using Python types. See https://github.com/RDFLib/rdflib/issues/1179.
    """
    return pd.DataFrame(
        data=([None if x is None else x.toPython() for x in row] for row in results),
        columns=[str(x) for x in results.vars],
    )

---

## load ranges data into dataframe

In [3]:
df = pd.read_csv('ASUoccurrence-new-cols.csv')
df.shape

(8962, 50)

---

## extract columns about length measurements

In [4]:
length_cols = ['ear_from_notch', 'hind_foot_with_claw'] + [col for col in df.columns if col.endswith('length')]
length_cols.sort()
length_cols

['ear_from_notch',
 'ear_length',
 'embryo_length',
 'forearm_length',
 'hind_foot_length',
 'hind_foot_with_claw',
 'tail_length',
 'testicle_length',
 'total_length',
 'tragus_length']

---

## calculate number of length and null values

In [5]:
length_count_df = pd.DataFrame(df[length_cols].count(), columns=['trait_count'])
length_na_count_df = pd.DataFrame(df[length_cols].isna().sum(), columns=['null_count'])
pd.concat([length_count_df, length_na_count_df], axis=1)

,trait_count,null_count
ear_from_notch,18,8944
ear_length,512,8450
embryo_length,1,8961
forearm_length,25,8937
hind_foot_length,520,8442
hind_foot_with_claw,10,8952
tail_length,528,8434
testicle_length,138,8824
total_length,525,8437
tragus_length,25,8937


### subset count testes lenght and width

In [6]:
testes_df = df[['testicle_length', 'testicle_width']]

In [7]:
testes_df.count()

testicle_length    138
testicle_width      43
dtype: int64

---

## load data into rdflib graph

In [8]:
g = rdflib.Graph()
g.parse('ranges-test-data.ttl')

<Graph identifier=Ne0a040cb8b984a2391a99165f2960c81 (<class 'rdflib.graph.Graph'>)>

----

## calculate number of trait length values

In [9]:
length_count_query = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX length: <http://purl.obolibrary.org/obo/PATO_0000122>

SELECT ?trait (COUNT(?value) AS ?trait_count) WHERE {
      ?observation ?trait ?value . 
      ?trait rdfs:subClassOf* length:  
}  
GROUP BY ?trait
ORDER BY ?trait
"""

In [10]:
length_count_results = sparql_results_to_df(g.query(length_count_query))
length_count_results.trait = length_count_results.trait.str.replace('http://purl.obolibrary.org/obo/FOVT/data#', 'FOVT:')
length_count_results

,trait,trait_count
0,FOVT:ear_from_notch,18
1,FOVT:ear_length,512
2,FOVT:embryo_length,1
3,FOVT:forearm_length,25
4,FOVT:hind_foot_length,520
5,FOVT:hind_foot_with_claw,10
6,FOVT:tail_length,528
7,FOVT:testicle_length,138
8,FOVT:total_length,525
9,FOVT:tragus_length,25


### compare counts in dataframe to query results

In [11]:
set(length_count_df.trait_count.values) == set(length_count_results.trait_count.values)

True

---

## calcluate number of null trait values

In [12]:
null_count_query = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX length: <http://purl.obolibrary.org/obo/PATO_0000122>
PREFIX traitvalue: <http://purl.obolibrary.org/obo/FOVT/data#>

SELECT ?trait (COUNT(?observation) AS ?null_count) where {
      ?observation traitvalue:occurrenceID ?id .
      ?trait rdfs:subClassOf* length:; rdf:type owl:DatatypeProperty  
      FILTER NOT EXISTS { ?observation ?trait ?value . }
} 
GROUP BY ?trait
ORDER BY ?trait
"""

In [13]:
null_count_results = sparql_results_to_df(g.query(null_count_query))
null_count_results.trait = length_count_results.trait.str.replace('http://purl.obolibrary.org/obo/FOVT/data#', 'FOVT:')
null_count_results

,trait,null_count
0,FOVT:ear_from_notch,8944
1,FOVT:ear_length,8450
2,FOVT:embryo_length,8961
3,FOVT:forearm_length,8937
4,FOVT:hind_foot_length,8442
5,FOVT:hind_foot_with_claw,8952
6,FOVT:tail_length,8434
7,FOVT:testicle_length,8824
8,FOVT:total_length,8437
9,FOVT:tragus_length,8937


### compare counts in dataframe to query results

In [14]:
set(length_na_count_df.null_count.values) == set(null_count_results.null_count.values)

True

---

## get counts of slots that are mapped to classes with the axiom *inheres in some testes*

In [15]:
inheres_in_count_query = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX traitvalue: <http://purl.obolibrary.org/obo/FOVT/data#>
PREFIX inheres_in: <http://purl.obolibrary.org/obo/RO_0000052>
PREFIX testes: <http://purl.obolibrary.org/obo/UBERON_0000473>

SELECT ?trait (COUNT(?value) AS ?trait_count) WHERE {
    # find traits/properties that have been punned as subclasses
    # of classes defined using the axiom "'inheres in' some testes"
    ?trait rdfs:subClassOf+ [
            owl:onProperty inheres_in:;
        	owl:someValuesFrom testes:
    ] .
    ?observation ?trait ?value . # retrieve values of traits
} GROUP BY ?trait
"""

In [16]:
testes_count_results = sparql_results_to_df(g.query(inheres_in_count_query))
testes_count_results.trait = testes_count_results.trait.str.replace('http://purl.obolibrary.org/obo/FOVT/data#', 'FOVT:')
testes_count_results

,trait,trait_count
0,FOVT:testicle_length,138
1,FOVT:testicle_width,43


### compare testes length trait counts from the testes dataframe to the results from sparql query

In [17]:
set(testes_df.count().values) == set(testes_count_results.trait_count.values)

True

---

## get data from slots that are mapped to classes with the axiom *inheres in some testes*

In [18]:
inheres_in_query = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX traitvalue: <http://purl.obolibrary.org/obo/FOVT/data#>
PREFIX inheres_in: <http://purl.obolibrary.org/obo/RO_0000052>
PREFIX testes: <http://purl.obolibrary.org/obo/UBERON_0000473>

SELECT ?trait ?value  WHERE {
    # find traits/properties that have been punned as subclasses
    # of classes defined using the axiom "'inheres is' some testes"
    ?trait rdfs:subClassOf+ [
            owl:onProperty inheres_in:;
        	owl:someValuesFrom testes:
    ] .
    ?observation ?trait ?value . # retrieve values of traits
}
"""

In [19]:
data_results = sparql_results_to_df(g.query(inheres_in_query))
data_results.trait = data_results.trait.str.replace('http://purl.obolibrary.org/obo/FOVT/data#', '')
data_results.head()

,trait,value
0,testicle_length,5.0
1,testicle_length,5.0
2,testicle_length,5.0
3,testicle_length,5.0
4,testicle_length,5.0


### compare testes length data from the testes dataframe to the results from sparql query

In [20]:
testes_length_set = set(testes_df.testicle_length.dropna().values)

In [21]:
testical_length_set = set(data_results[data_results.trait == 'testicle_length']['value'].values)

In [22]:
testes_length_set == testical_length_set

True

### compare testes width data from the testes dataframe to the results from sparql query

In [23]:
testes_width_set = set(testes_df.testicle_width.dropna().values)

In [24]:
testical_width_set = set(data_results[data_results.trait == 'testicle_width']['value'].values)

In [25]:
testes_width_set == testical_width_set

True